# Interpretabilidad funcional del Enfoque A

Este cuaderno reconstruye, sobre el dominio continuo de la frecuencia, la función de peso de una regresión logística aplicada a la representación **A1 + Cholesky**, con el fin de identificar qué regiones espectrales contribuyen de forma estable a la clasificación.

La elección de esta configuración es deliberada:

- **A1** mantiene una observación por sujeto y evita mezclar la interpretación con el efecto de entrenar por épocas de A2.
- **Cholesky** fue la representación A1 con mejor desempeño observado con regresión logística y preserva explícitamente la geometría funcional $L^2$.
- Se evita FPCA en este análisis principal para que la reconstrucción sea directa y no requiera deshacer componentes que cambian entre pliegues.

Para cada canal $c$ se reconstruye una función de peso

$$
\beta_c(f)=\sum_{k=1}^{K} b_{ck}\,\phi_k(f),
$$

a partir de los coeficientes de la regresión logística ajustada en cada uno de los **15 pliegues externos** (5 folds × 3 semillas). La variabilidad entre pliegues se resume de forma **descriptiva** mediante media y desviación estándar; los pliegues repetidos no se tratan como observaciones independientes para inferencia. Además, se incorpora un diagnóstico específico de sensibilidad en los bordes de la base B-spline para evitar interpretar como patrón neurofisiológico una amplificación localizada cerca de 4 o 13 Hz.

**Interpretación:** la clase positiva es `PD_OFF`. Un valor positivo de $\beta_c(f)$ indica que, manteniendo las demás variables del modelo constantes, un aumento de `log10(PSD relativa)` alrededor de esa frecuencia incrementa el log-odds predicho de `PD_OFF`; un valor negativo actúa en la dirección opuesta. Estas funciones son pesos multivariados regularizados, no efectos causales ni pruebas univariadas de diferencia entre grupos.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import json
import logging
import platform
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy import linalg
from scipy.integrate import simpson
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import SplineTransformer, StandardScaler

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
    force=True,
)
LOGGER = logging.getLogger('interpretabilidad_ADF')

CONFIG: dict[str, Any] = {
    'random_state': 42,
    'seeds': [42, 123, 2024],
    'n_folds': 5,
    'logreg_C': 1.0,
    'logreg_max_iter': 2000,
    'bspline_degree': 3,
    'gram_grid_points': 1001,
    'plot_grid_points': 301,
    'expected_channels': ['C3', 'Cz', 'C4', 'P7', 'P8', 'P3', 'Pz', 'P4', 'O1', 'O2'],
    'class_positive': 'PD_OFF',
}

np.random.seed(CONFIG['random_state'])

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

## 1. Entradas y carpeta de salida

El cuaderno utiliza el dataset A1 generado por el cuaderno `02A_UNM_enfoque_A.ipynb`. No reutiliza coeficientes
de modelos entrenados anteriormente: repite la regresión logística sobre las mismas 15 particiones externas para
poder almacenar y reconstruir los pesos de cada pliegue. Si está disponible, utiliza además el resumen del
cuaderno `04_UNM_aporte_ADF.ipynb` para verificar la consistencia del desempeño.

In [ ]:
# Rutas portátiles: Google Drive solo se monta si USE_GOOGLE_DRIVE=1 (Colab).
import os
from pathlib import Path

if os.environ.get('USE_GOOGLE_DRIVE') == '1':
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
else:
    print('Ejecución local: se usan PARKINSON_EEG_ROOT y PARKINSON_EEG_DATA.')


_REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FINAL_ROOT = Path(os.environ.get('PARKINSON_EEG_ROOT', _REPO_ROOT / 'artifacts')).expanduser().resolve()
A1_PATH = FINAL_ROOT / '02A_enfoque_A' / 'enfoque_A_sujeto_dataset.npz'
APORTE_SUMMARY_PATH = FINAL_ROOT / '04_aporte_ADF' / 'resumen_LogReg.csv'

OUT_DIR = FINAL_ROOT / '05_interpretabilidad'
FIG_DIR = OUT_DIR / 'figuras'

if not A1_PATH.is_file():
    raise FileNotFoundError(
        f'No se encuentra el dataset A1: {A1_PATH}. '
        'Ejecute primero el cuaderno 02A.'
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = OUT_DIR / 'config_interpretabilidad.json'
CONFIG_PATH.write_text(
    json.dumps(CONFIG, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('Entrada A1:', A1_PATH)
print('Salida:', OUT_DIR)
print('Resumen del cuaderno 04 disponible:', APORTE_SUMMARY_PATH.exists())

## 2. Carga y validación del Enfoque A1

La representación A1 contiene una función espectral por sujeto y canal. `X_raw` almacena los coeficientes B-spline y `X_chol` las coordenadas obtenidas mediante la transformación de Cholesky. Ambas describen la misma función en sistemas de coordenadas diferentes.

In [ ]:
data = np.load(A1_PATH, allow_pickle=False)

required_keys = {'X_raw', 'X_chol', 'y', 'groups', 'channels', 'freqs', 'k_opt', 'band'}
missing = required_keys.difference(data.files)
if missing:
    raise KeyError(f'Faltan claves en {A1_PATH.name}: {sorted(missing)}')

X_raw = np.asarray(data['X_raw'], dtype=float)
X_chol = np.asarray(data['X_chol'], dtype=float)
y = np.asarray(data['y'], dtype=int)
groups = np.asarray(data['groups'])
channels = [str(v) for v in data['channels']]
freqs_obs = np.asarray(data['freqs'], dtype=float)
k_opt = int(np.asarray(data['k_opt']).item())
band = np.asarray(data['band'], dtype=float)

n_subjects, n_channels, n_basis = X_raw.shape

assert X_chol.shape == X_raw.shape
assert len(y) == n_subjects == len(groups)
assert len(np.unique(groups)) == n_subjects, 'A1 debe contener una observación por sujeto.'
assert set(np.unique(y)) == {0, 1}
assert int((y == 0).sum()) == 24 and int((y == 1).sum()) == 24
assert channels == CONFIG['expected_channels'], (channels, CONFIG['expected_channels'])
assert np.isfinite(X_raw).all() and np.isfinite(X_chol).all()

print('X_raw :', X_raw.shape)
print('X_chol:', X_chol.shape)
print('Sujetos:', n_subjects, '| HC:', int((y == 0).sum()), '| PD_OFF:', int((y == 1).sum()))
print('Canales:', channels)
print('Frecuencias observadas:', freqs_obs[0], 'a', freqs_obs[-1], 'Hz | n =', len(freqs_obs))
print('k óptimo:', k_opt, '| funciones base:', n_basis)
print('Banda funcional:', band.tolist())

## 3. Reconstrucción exacta de la base, matriz de Gram y Cholesky

El cuaderno 02A construyó `X_chol` como

$$
\mathbf z = \mathbf cL, \qquad G=LL^\top,
$$

donde $\mathbf c$ son los coeficientes B-spline. Aquí se reconstruyen la misma base y la misma matriz de Gram a partir de los metadatos guardados y se verifica numéricamente que

$$
X_{\mathrm{raw}}L \approx X_{\mathrm{chol}}.
$$

Esta comprobación es crítica antes de transformar los coeficientes de la regresión de vuelta al dominio funcional.

In [ ]:
BSPLINE_DEGREE = CONFIG['bspline_degree']
BAND_FMIN, BAND_FMAX = float(band[0]), float(band[1])

spline = SplineTransformer(
    n_knots=k_opt,
    degree=BSPLINE_DEGREE,
    include_bias=True,
)
B_obs = spline.fit_transform(freqs_obs.reshape(-1, 1))

if B_obs.shape[1] != n_basis:
    raise ValueError(
        f'La base reconstruida tiene {B_obs.shape[1]} funciones, pero X_raw tiene {n_basis}.'
    )

freqs_gram = np.linspace(
    BAND_FMIN,
    BAND_FMAX,
    CONFIG['gram_grid_points'],
)
B_gram = spline.transform(freqs_gram.reshape(-1, 1))
products = B_gram[:, :, None] * B_gram[:, None, :]
G = simpson(products, x=freqs_gram, axis=0)
G = 0.5 * (G + G.T)
L = np.linalg.cholesky(G)

reconstruction_error = float(np.max(np.abs(X_raw @ L - X_chol)))
print(f'Error máximo |X_raw @ L - X_chol| = {reconstruction_error:.3e}')

if reconstruction_error > 1e-10:
    raise RuntimeError(
        'La reconstrucción de Cholesky no coincide con el dataset. '
        'No continuar con la interpretación hasta resolver esta discrepancia.'
    )

freq_grid = np.linspace(
    BAND_FMIN,
    BAND_FMAX,
    CONFIG['plot_grid_points'],
)
B_grid = spline.transform(freq_grid.reshape(-1, 1))

print('Gram simétrica:', np.allclose(G, G.T, atol=1e-12))
print('Autovalor mínimo de G:', float(np.linalg.eigvalsh(G).min()))
print('Malla de interpretación:', len(freq_grid), 'puntos')

## 3.1. Diagnóstico de sensibilidad en los bordes de la base B-spline

Las B-splines tienen soporte local y, en los extremos del dominio, las funciones base pueden tener una norma $L^2$ menor que las funciones interiores. Al reconstruir la función de peso mediante $L^{-T}$, ciertas direcciones localizadas en los bordes pueden adquirir amplitudes puntuales grandes aun cuando su contribución integrada al logit sea moderada.

Por ello, antes de interpretar picos cercanos a 4 o 13 Hz se cuantifica:

- el número de condición de la matriz de Gram;
- la norma $L^2$ de cada función base;
- una **región interior definida por la propia malla de nudos**, excluyendo un intervalo de nudos en cada extremo.

Este análisis no modifica el modelo ni elimina frecuencias del entrenamiento. Es únicamente un control de sensibilidad para la **interpretación** de $\beta_c(f)$.


In [ ]:
# Diagnóstico objetivo de borde a partir de los nudos de la base.
core_knots = np.asarray(
    spline.bsplines_[0].t[BSPLINE_DEGREE:-BSPLINE_DEGREE],
    dtype=float,
)
if len(core_knots) != k_opt:
    raise RuntimeError(
        f'Se esperaban {k_opt} nudos principales y se obtuvieron {len(core_knots)}.'
    )

knot_spacing = float(np.median(np.diff(core_knots)))
INTERIOR_FMIN = float(core_knots[1])
INTERIOR_FMAX = float(core_knots[-2])

basis_l2_norms = np.sqrt(np.diag(G))
gram_condition_number = float(np.linalg.cond(G))

basis_diag_df = pd.DataFrame({
    'indice_base': np.arange(1, n_basis + 1),
    'norma_L2_base': basis_l2_norms,
})
basis_diag_path = OUT_DIR / 'diagnostico_base_bspline.csv'
basis_diag_df.to_csv(basis_diag_path, index=False)

print('Nudos principales:', np.round(core_knots, 4).tolist())
print(f'Espaciado de nudos: {knot_spacing:.4f} Hz')
print(
    'Región interior para sensibilidad de borde:',
    f'{INTERIOR_FMIN:.3f}–{INTERIOR_FMAX:.3f} Hz',
)
print(f'Número de condición de G: {gram_condition_number:.1f}')
print(
    'Norma L2 de bases de borde / mediana interior:',
    f'{basis_l2_norms[0]:.4f} / {np.median(basis_l2_norms[2:-2]):.4f}',
    'y',
    f'{basis_l2_norms[-1]:.4f} / {np.median(basis_l2_norms[2:-2]):.4f}',
)
display(basis_diag_df.round(5))


## 4. De pesos de LogReg a función de peso $\beta_c(f)$

La regresión se ajusta sobre las coordenadas de Cholesky estandarizadas. Si $a_c$ es el vector de pesos del modelo sobre las coordenadas Cholesky **después de deshacer la estandarización**, entonces

$$
\mathbf z_c = \mathbf c_cL,
$$

y el término lineal del modelo es

$$
\mathbf z_c a_c = \mathbf c_cLa_c.
$$

Para expresarlo como un producto interno funcional

$$
\langle x_c,\beta_c\rangle_{L^2}
=\mathbf c_cG\mathbf b_c
=\mathbf c_cLL^\top\mathbf b_c,
$$

debe cumplirse

$$
a_c=L^\top\mathbf b_c,
\qquad
\mathbf b_c=L^{-\top}a_c.
$$

Finalmente,

$$
\beta_c(f)=\sum_k b_{ck}\phi_k(f).
$$

El cuaderno verifica numéricamente, en cada pliegue, que el logit reconstruido mediante el producto interno funcional coincide con el `decision_function` de `scikit-learn`.

In [ ]:
def compute_metrics(y_true: np.ndarray, y_proba: np.ndarray) -> dict[str, float]:
    """Métricas binarias a nivel de sujeto con PD_OFF como clase positiva."""
    y_pred = (y_proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    try:
        auc = float(roc_auc_score(y_true, y_proba))
    except ValueError:
        auc = np.nan
    return {
        'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'auc': auc,
    }


def fit_logreg_with_scaler(
    X_train: np.ndarray,
    y_train: np.ndarray,
) -> tuple[StandardScaler, LogisticRegression, np.ndarray]:
    """Ajusta StandardScaler + LogReg exactamente como en los cuadernos 03 y 04."""
    flat = X_train.reshape(len(X_train), -1)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(flat)
    model = LogisticRegression(
        C=CONFIG['logreg_C'],
        class_weight='balanced',
        max_iter=CONFIG['logreg_max_iter'],
        random_state=CONFIG['random_state'],
    )
    model.fit(X_scaled, y_train)
    return scaler, model, X_scaled


def destandardize_logreg(
    scaler: StandardScaler,
    model: LogisticRegression,
) -> tuple[np.ndarray, float]:
    """Devuelve pesos e intercepto equivalentes sobre X_chol sin estandarizar."""
    w_std = np.asarray(model.coef_[0], dtype=float)
    w_chol = w_std / scaler.scale_
    intercept_chol = float(
        model.intercept_[0] - np.dot(w_std, scaler.mean_ / scaler.scale_)
    )
    return w_chol, intercept_chol


def chol_weights_to_functional_beta(
    w_chol: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Convierte pesos sobre z=cL en coeficientes de beta y beta(f).

    Returns
    -------
    beta_basis : (n_channels, n_basis)
        Coeficientes B-spline de la función de peso.
    beta_grid : (n_channels, n_freq_grid)
        Función de peso evaluada sobre la malla densa.
    """
    a = w_chol.reshape(n_channels, n_basis)
    beta_basis = np.empty_like(a)
    for ch in range(n_channels):
        beta_basis[ch] = linalg.solve_triangular(
            L.T,
            a[ch],
            lower=False,
            check_finite=True,
        )
    beta_grid = beta_basis @ B_grid.T
    return beta_basis, beta_grid


def functional_logit(
    X_raw_values: np.ndarray,
    beta_basis: np.ndarray,
    intercept: float,
) -> np.ndarray:
    """Calcula b0 + sum_c <x_c, beta_c>_L2 desde coeficientes B-spline."""
    functional_part = np.einsum(
        'nck,kl,cl->n',
        X_raw_values,
        G,
        beta_basis,
        optimize=True,
    )
    return functional_part + intercept


def mean_pairwise_correlation(curves: np.ndarray) -> float:
    """Correlación media entre curvas de distintos pliegues; descriptiva."""
    corr = np.corrcoef(curves)
    upper = corr[np.triu_indices_from(corr, k=1)]
    upper = upper[np.isfinite(upper)]
    return float(upper.mean()) if len(upper) else np.nan

## 5. Ajuste en las 15 particiones externas y reconstrucción de pesos

Todas las transformaciones dependientes de los datos se ajustan **solo con el entrenamiento externo**. Para cada pliegue se guardan:

- métricas del conjunto de prueba;
- probabilidades predichas por sujeto;
- pesos de LogReg en coordenadas estandarizadas y desestandarizadas;
- coeficientes B-spline de $\beta_c$;
- $\beta_c(f)$ sobre una malla común;
- errores de equivalencia numérica entre las tres expresiones del logit.

La reconstrucción se considera válida solo si las tres formas del logit coinciden numéricamente.

In [ ]:
metric_records: list[dict[str, Any]] = []
prediction_records: list[dict[str, Any]] = []
split_records: list[dict[str, Any]] = []
transform_records: list[dict[str, Any]] = []

beta_basis_folds: list[np.ndarray] = []
beta_grid_folds: list[np.ndarray] = []
w_chol_folds: list[np.ndarray] = []
seed_fold_pairs: list[tuple[int, int]] = []

for seed in CONFIG['seeds']:
    splitter = StratifiedGroupKFold(
        n_splits=CONFIG['n_folds'],
        shuffle=True,
        random_state=seed,
    )

    for fold, (idx_train, idx_test) in enumerate(
        splitter.split(X_chol, y, groups)
    ):
        train_groups = set(groups[idx_train])
        test_groups = set(groups[idx_test])
        if train_groups & test_groups:
            raise AssertionError('Fuga de sujetos entre train y test.')

        scaler, model, X_train_scaled = fit_logreg_with_scaler(
            X_chol[idx_train],
            y[idx_train],
        )
        X_test_flat = X_chol[idx_test].reshape(len(idx_test), -1)
        X_test_scaled = scaler.transform(X_test_flat)
        probability = model.predict_proba(X_test_scaled)[:, 1]

        metrics = compute_metrics(y[idx_test], probability)
        metrics.update({'seed': seed, 'fold': fold})
        metric_records.append(metrics)

        w_chol, intercept_chol = destandardize_logreg(scaler, model)
        beta_basis, beta_grid_fold = chol_weights_to_functional_beta(w_chol)

        # ------------------------------------------------------------------
        # Validación algebraica de la reconstrucción
        # ------------------------------------------------------------------
        logit_sklearn = model.decision_function(X_test_scaled)
        logit_chol = X_test_flat @ w_chol + intercept_chol
        logit_functional = functional_logit(
            X_raw[idx_test], beta_basis, intercept_chol
        )

        err_chol = float(np.max(np.abs(logit_sklearn - logit_chol)))
        err_functional = float(np.max(np.abs(logit_sklearn - logit_functional)))

        if err_chol > 1e-8 or err_functional > 1e-8:
            raise RuntimeError(
                f'Fallo de equivalencia en seed={seed}, fold={fold}: '
                f'err_chol={err_chol:.3e}, err_func={err_functional:.3e}'
            )

        beta_basis_folds.append(beta_basis)
        beta_grid_folds.append(beta_grid_fold)
        w_chol_folds.append(w_chol.reshape(n_channels, n_basis))
        seed_fold_pairs.append((seed, fold))

        transform_records.append({
            'seed': seed,
            'fold': fold,
            'n_train': int(len(idx_train)),
            'n_test': int(len(idx_test)),
            'max_error_logit_chol': err_chol,
            'max_error_logit_functional': err_functional,
            'intercept_original_chol': intercept_chol,
        })

        for role, indices in [('train', idx_train), ('test', idx_test)]:
            for idx in indices:
                split_records.append({
                    'seed': seed,
                    'fold': fold,
                    'rol': role,
                    'group': str(groups[idx]),
                    'label': int(y[idx]),
                    'grupo': 'PD_OFF' if y[idx] == 1 else 'HC',
                })

        for local_pos, idx in enumerate(idx_test):
            prediction_records.append({
                'seed': seed,
                'fold': fold,
                'group': str(groups[idx]),
                'y_true': int(y[idx]),
                'grupo': 'PD_OFF' if y[idx] == 1 else 'HC',
                'prob_PD_OFF': float(probability[local_pos]),
            })

metric_df = pd.DataFrame(metric_records)
prediction_df = pd.DataFrame(prediction_records)
splits_df = pd.DataFrame(split_records)
transform_df = pd.DataFrame(transform_records)

beta_basis_arr = np.stack(beta_basis_folds, axis=0)
beta_grid_arr = np.stack(beta_grid_folds, axis=0)
w_chol_arr = np.stack(w_chol_folds, axis=0)
seed_fold_arr = np.asarray(seed_fold_pairs, dtype=int)

assert beta_basis_arr.shape == (15, n_channels, n_basis)
assert beta_grid_arr.shape == (15, n_channels, len(freq_grid))
assert len(metric_df) == 15

metric_path = OUT_DIR / 'metricas_por_fold.csv'
prediction_path = OUT_DIR / 'predicciones_por_sujeto.csv'
split_path = OUT_DIR / 'particiones.csv'
transform_path = OUT_DIR / 'validacion_reconstruccion.csv'
beta_path = OUT_DIR / 'funciones_peso.npz'

metric_df.to_csv(metric_path, index=False)
prediction_df.to_csv(prediction_path, index=False)
splits_df.to_csv(split_path, index=False)
transform_df.to_csv(transform_path, index=False)
np.savez_compressed(
    beta_path,
    beta_basis=beta_basis_arr,
    beta_grid=beta_grid_arr,
    w_chol=w_chol_arr,
    freq_grid=freq_grid,
    channels=np.asarray(channels),
    seed_fold=seed_fold_arr,
    gram=G,
    cholesky=L,
)

print('Evaluaciones:', len(metric_df))
print('Máx. error logit (Cholesky):', f"{transform_df['max_error_logit_chol'].max():.3e}")
print('Máx. error logit (funcional):', f"{transform_df['max_error_logit_functional'].max():.3e}")
display(metric_df.round(4))

## 6. Consistencia del desempeño

Antes de interpretar los pesos se verifica que esta repetición de **A1 + Cholesky + LogReg** reproduzca el desempeño obtenido en el cuaderno 04 (análisis de aporte del ADF). Si el resumen de ese cuaderno está disponible, la comparación se realiza automáticamente.

In [ ]:
METRICS = ['balanced_acc', 'sensitivity', 'specificity', 'f1', 'auc']

summary_rows = []
for metric in METRICS:
    summary_rows.append({
        'metrica': metric,
        'media': float(metric_df[metric].mean()),
        'de': float(metric_df[metric].std(ddof=1)),
    })
summary_metrics = pd.DataFrame(summary_rows)
summary_metric_path = OUT_DIR / 'resumen_metricas.csv'
summary_metrics.to_csv(summary_metric_path, index=False)

display(summary_metrics.round(4))
print(
    'A1 Cholesky + LogReg | Balanced accuracy = '
    f"{metric_df['balanced_acc'].mean():.3f} ± {metric_df['balanced_acc'].std(ddof=1):.3f} | "
    'AUC = '
    f"{metric_df['auc'].mean():.3f} ± {metric_df['auc'].std(ddof=1):.3f}"
)

trace_rows = []
if APORTE_SUMMARY_PATH.exists():
    aporte = pd.read_csv(APORTE_SUMMARY_PATH)
    match = aporte[aporte['metodo'] == 'A1 Cholesky']
    if len(match) == 1:
        row_ref = match.iloc[0]
        for metric in METRICS:
            current = float(metric_df[metric].mean())
            previous = float(row_ref[f'{metric}_mean'])
            trace_rows.append({
                'metrica': metric,
                'cuaderno_04': previous,
                'este_cuaderno': current,
                'diferencia': current - previous,
            })
        trace_df = pd.DataFrame(trace_rows)
        trace_path = OUT_DIR / 'consistencia_con_aporte_ADF.csv'
        trace_df.to_csv(trace_path, index=False)
        print('\nConsistencia con el cuaderno 04:')
        display(trace_df.round(10))
        if np.max(np.abs(trace_df['diferencia'])) > 1e-8:
            print(
                'ADVERTENCIA: no se reproduce exactamente el resultado del cuaderno 04. '
                'Revisar particiones/configuración antes de interpretar las funciones de peso.'
            )
        else:
            print('Desempeño idéntico al del cuaderno 04 dentro de la tolerancia numérica.')
    else:
        print('No se encontró una única fila "A1 Cholesky" en el resumen del cuaderno 04.')
else:
    print('Resumen del cuaderno 04 no disponible; se omite la verificación de consistencia.')

## 7. Estabilidad de las funciones de peso entre pliegues

La estabilidad se describe mediante:

- norma funcional $L^2$ de $\beta_c$ por canal;
- correlación media de la **forma** de $\beta_c(f)$ entre pares de pliegues;
- proporción de la malla frecuencial en la que al menos el 80 % de los pliegues comparte el mismo signo.

Estas cantidades son diagnósticos descriptivos. En especial, la estabilidad de signo puede ser poco informativa en regiones donde $\beta_c(f)$ está muy cerca de cero.

In [ ]:
beta_mean = beta_grid_arr.mean(axis=0)
beta_sd = beta_grid_arr.std(axis=0, ddof=1)

# Norma L2 exacta desde los coeficientes de la base y la matriz de Gram.
l2_norms = np.sqrt(
    np.einsum('fck,kl,fcl->fc', beta_basis_arr, G, beta_basis_arr, optimize=True)
)

sign_positive_fraction = (beta_grid_arr > 0).mean(axis=0)
sign_consistency = np.maximum(sign_positive_fraction, 1.0 - sign_positive_fraction)

stability_rows = []
for ch_idx, ch in enumerate(channels):
    stability_rows.append({
        'canal': ch,
        'norma_L2_media': float(l2_norms[:, ch_idx].mean()),
        'norma_L2_de': float(l2_norms[:, ch_idx].std(ddof=1)),
        'correlacion_media_entre_pliegues': mean_pairwise_correlation(
            beta_grid_arr[:, ch_idx, :]
        ),
        'proporcion_frecuencias_signo_consistente_80': float(
            (sign_consistency[ch_idx] >= 0.80).mean()
        ),
    })

stability_df = pd.DataFrame(stability_rows)
stability_path = OUT_DIR / 'estabilidad_funciones_peso.csv'
stability_df.to_csv(stability_path, index=False)

display(stability_df.round(4))

## 8. Resumen descriptivo por bandas neurofisiológicas

La interpretación principal se realiza sobre la **función completa** $\beta_c(f)$. Como resumen secundario guiado por la literatura se consideran:

- theta completa: 4–8 Hz;
- theta lenta: 4–5.5 Hz;
- pre-alfa o theta rápida: 5.5–8 Hz;
- alfa: 8–13 Hz.

La subdivisión de theta no reemplaza el análisis de 4–8 Hz. Se utiliza para determinar si un patrón observado en theta es relativamente homogéneo o está concentrado en una subregión. En particular, el intervalo 5.5–8 Hz se analiza porque ha sido descrito en la literatura de EEG en enfermedad de Parkinson como **theta rápida / pre-alfa**, con hallazgos reproducibles en regiones posteriores.

Se reporta la integral firmada de $\beta_c(f)$ y su variabilidad entre pliegues. La dirección de esta integral **no debe confundirse con una diferencia univariada de potencia entre grupos**: sigue siendo un peso condicional del modelo multivariado.


In [ ]:
BANDS = {
    'theta_4_8': (4.0, 8.0),
    'theta_lenta_4_5_5': (4.0, 5.5),
    'pre_alfa_5_5_8': (5.5, 8.0),
    'alfa_8_13': (8.0, 13.0),
}

band_fold_rows = []
for fold_idx, (seed, fold) in enumerate(seed_fold_pairs):
    for ch_idx, ch in enumerate(channels):
        curve = beta_grid_arr[fold_idx, ch_idx]
        for band_name, (fmin, fmax) in BANDS.items():
            mask = (freq_grid >= fmin) & (freq_grid <= fmax)
            signed_integral = float(simpson(curve[mask], x=freq_grid[mask]))
            absolute_integral = float(simpson(np.abs(curve[mask]), x=freq_grid[mask]))
            band_fold_rows.append({
                'seed': seed,
                'fold': fold,
                'canal': ch,
                'banda': band_name,
                'beta_media': float(curve[mask].mean()),
                'integral_beta': signed_integral,
                'integral_abs_beta': absolute_integral,
            })

band_fold_df = pd.DataFrame(band_fold_rows)
band_fold_path = OUT_DIR / 'beta_por_banda_y_fold.csv'
band_fold_df.to_csv(band_fold_path, index=False)

band_summary = (
    band_fold_df
    .groupby(['canal', 'banda'], as_index=False)
    .agg(
        beta_media=('beta_media', 'mean'),
        beta_media_de=('beta_media', 'std'),
        integral_beta_media=('integral_beta', 'mean'),
        integral_beta_de=('integral_beta', 'std'),
        integral_abs_media=('integral_abs_beta', 'mean'),
        integral_abs_de=('integral_abs_beta', 'std'),
        proporcion_integral_positivo=('integral_beta', lambda s: float((s > 0).mean())),
        proporcion_integral_negativo=('integral_beta', lambda s: float((s < 0).mean())),
    )
)
band_summary_path = OUT_DIR / 'resumen_beta_por_banda.csv'
band_summary.to_csv(band_summary_path, index=False)

display(band_summary.round(4))

## 8.1. Sensibilidad de la interpretación al borde superior de 13 Hz

La figura completa puede mostrar amplitudes puntuales grandes cerca de 13 Hz. Estas amplitudes se conservan en los productos completos —no se ocultan ni se eliminan—, pero se evalúan por separado porque coinciden con la zona de soporte de las funciones B-spline de borde y pueden dominar visualmente la escala.

Se comparan las magnitudes máximas en la región interior y en los dos bordes definidos por los nudos. Además, para alfa se calcula un resumen de sensibilidad limitado a $8$–$f_{\mathrm{interior,max}}$ Hz. Este último **no constituye una nueva banda fisiológica** ni sustituye a alfa 8–13 Hz; solo permite comprobar si una conclusión depende del extremo de la base.


In [ ]:
interior_mask = (freq_grid >= INTERIOR_FMIN) & (freq_grid <= INTERIOR_FMAX)
lower_edge_mask = freq_grid < INTERIOR_FMIN
upper_edge_mask = freq_grid > INTERIOR_FMAX

edge_rows = []
for ch_idx, ch in enumerate(channels):
    curves = beta_grid_arr[:, ch_idx, :]
    max_interior = np.max(np.abs(curves[:, interior_mask]), axis=1)
    max_lower = np.max(np.abs(curves[:, lower_edge_mask]), axis=1)
    max_upper = np.max(np.abs(curves[:, upper_edge_mask]), axis=1)

    edge_rows.append({
        'canal': ch,
        'max_abs_interior_media': float(max_interior.mean()),
        'max_abs_borde_inferior_media': float(max_lower.mean()),
        'ratio_borde_inferior_vs_interior': float(
            np.mean(max_lower / np.maximum(max_interior, 1e-12))
        ),
        'max_abs_borde_superior_media': float(max_upper.mean()),
        'ratio_borde_superior_vs_interior': float(
            np.mean(max_upper / np.maximum(max_interior, 1e-12))
        ),
        'correlacion_media_pliegues_region_interior': mean_pairwise_correlation(
            curves[:, interior_mask]
        ),
        'proporcion_signo_consistente_80_region_interior': float(
            (sign_consistency[ch_idx, interior_mask] >= 0.80).mean()
        ),
    })

edge_sensitivity_df = pd.DataFrame(edge_rows)
edge_sensitivity_path = OUT_DIR / 'sensibilidad_borde_funciones_peso.csv'
edge_sensitivity_df.to_csv(edge_sensitivity_path, index=False)

display(edge_sensitivity_df.round(4))

# Sensibilidad específica para alfa sin el último intervalo de nudos.
alpha_interior_rows = []
alpha_interior_mask = (freq_grid >= 8.0) & (freq_grid <= INTERIOR_FMAX)
for fold_idx, (seed, fold) in enumerate(seed_fold_pairs):
    for ch_idx, ch in enumerate(channels):
        curve = beta_grid_arr[fold_idx, ch_idx]
        alpha_interior_rows.append({
            'seed': seed,
            'fold': fold,
            'canal': ch,
            'banda': f'alfa_interior_8_{INTERIOR_FMAX:.3f}',
            'beta_media': float(curve[alpha_interior_mask].mean()),
            'integral_beta': float(
                simpson(curve[alpha_interior_mask], x=freq_grid[alpha_interior_mask])
            ),
            'integral_abs_beta': float(
                simpson(
                    np.abs(curve[alpha_interior_mask]),
                    x=freq_grid[alpha_interior_mask],
                )
            ),
        })

alpha_interior_df = pd.DataFrame(alpha_interior_rows)
alpha_interior_summary = (
    alpha_interior_df
    .groupby(['canal', 'banda'], as_index=False)
    .agg(
        beta_media=('beta_media', 'mean'),
        beta_media_de=('beta_media', 'std'),
        integral_beta_media=('integral_beta', 'mean'),
        integral_beta_de=('integral_beta', 'std'),
        integral_abs_media=('integral_abs_beta', 'mean'),
        integral_abs_de=('integral_abs_beta', 'std'),
        proporcion_integral_positivo=('integral_beta', lambda s: float((s > 0).mean())),
        proporcion_integral_negativo=('integral_beta', lambda s: float((s < 0).mean())),
    )
)
alpha_interior_path = OUT_DIR / 'sensibilidad_alfa_sin_borde.csv'
alpha_interior_summary.to_csv(alpha_interior_path, index=False)

print(
    f'Control de sensibilidad alfa: 8–{INTERIOR_FMAX:.3f} Hz '
    '(no es una banda fisiológica nueva).'
)
display(alpha_interior_summary.round(4))


## 9. Funciones de peso por canal - vista completa

Cada panel muestra las 15 funciones estimadas, la media entre pliegues y una banda de media ± 1 DE. La vista completa 4–13 Hz se conserva como diagnóstico y permite comprobar cualquier amplificación de borde.

Las líneas verticales en 5.5 Hz y 8 Hz delimitan theta lenta, pre-alfa y alfa. Esta figura muestra **estabilidad/desestabilidad del patrón funcional**, no un intervalo de confianza inferencial.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15.5, 6.6), sharex=True)
axes = axes.ravel()

for ch_idx, (ax, ch) in enumerate(zip(axes, channels)):
    curves = beta_grid_arr[:, ch_idx, :]
    mean_curve = beta_mean[ch_idx]
    sd_curve = beta_sd[ch_idx]

    for curve in curves:
        ax.plot(freq_grid, curve, lw=0.8, alpha=0.20, color='0.45')
    ax.plot(freq_grid, mean_curve, lw=2.0, color='black', label='Media')
    ax.fill_between(
        freq_grid,
        mean_curve - sd_curve,
        mean_curve + sd_curve,
        alpha=0.16,
        color='0.35',
        label='Media ± 1 DE',
    )
    ax.axhline(0.0, lw=0.8, ls='--', color='0.55')
    ax.axvline(5.5, lw=0.8, ls=':', color='0.55')
    ax.axvline(8.0, lw=0.8, ls=':', color='0.55')
    ax.set_title(ch, fontsize=10)
    ax.grid(alpha=0.12)
    ax.spines[['top', 'right']].set_visible(False)

for ax in axes[5:]:
    ax.set_xlabel('Frecuencia (Hz)')
for ax in axes[::5]:
    ax.set_ylabel(r'Peso funcional $\beta_c(f)$')

axes[0].legend(frameon=True, fontsize=8, loc='best')
fig.tight_layout()

fig_weight_path = FIG_DIR / 'funciones_peso_completa.png'
fig.savefig(fig_weight_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', fig_weight_path)

## 10. Mapa resumen de la función de peso media

Este mapa es un complemento visual. Los colores representan el signo y magnitud de la función de peso media entre los 15 pliegues. Debe interpretarse junto con la figura anterior: una magnitud media alta no es suficiente si el signo o la forma son inestables entre pliegues.

In [ ]:
vmax = float(np.max(np.abs(beta_mean)))
if vmax == 0:
    vmax = 1.0
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
im = ax.imshow(
    beta_mean,
    aspect='auto',
    origin='upper',
    extent=[freq_grid[0], freq_grid[-1], n_channels - 0.5, -0.5],
    cmap='coolwarm',
    norm=norm,
)
ax.set_yticks(np.arange(n_channels))
ax.set_yticklabels(channels)
ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Canal')
ax.axvline(5.5, lw=0.9, ls=':', color='black', alpha=0.6)
ax.axvline(8.0, lw=0.9, ls=':', color='black', alpha=0.6)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'Peso funcional medio $\beta_c(f)$')
fig.tight_layout()

fig_map_path = FIG_DIR / 'mapa_funcion_peso_media.png'
fig.savefig(fig_map_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura guardada:', fig_map_path)

## 10.1. Vista focalizada en theta y pre-alfa

Para que las oscilaciones cercanas a 13 Hz no dominen la escala visual, se genera además una vista focalizada entre 4 y 10 Hz. **No se recortan los datos del modelo ni se reemplaza la figura completa**: esta es una figura de lectura, destinada a examinar con mayor resolución la región theta/pre-alfa que fue definida a priori a partir de la literatura.

La interpretación cuantitativa debe apoyarse en las integrales por banda y en su estabilidad entre pliegues, no únicamente en la amplitud visual de una curva.


In [ ]:
FOCUS_FMIN = 4.0
FOCUS_FMAX = 10.0
focus_mask = (freq_grid >= FOCUS_FMIN) & (freq_grid <= FOCUS_FMAX)

# Paneles por canal, focalizados en theta/pre-alfa.
fig, axes = plt.subplots(2, 5, figsize=(15.5, 6.6), sharex=True)
axes = axes.ravel()

for ch_idx, (ax, ch) in enumerate(zip(axes, channels)):
    curves = beta_grid_arr[:, ch_idx, :]
    mean_curve = beta_mean[ch_idx]
    sd_curve = beta_sd[ch_idx]

    for curve in curves:
        ax.plot(freq_grid[focus_mask], curve[focus_mask], lw=0.8, alpha=0.20, color='0.45')
    ax.plot(
        freq_grid[focus_mask],
        mean_curve[focus_mask],
        lw=2.0,
        color='black',
        label='Media',
    )
    ax.fill_between(
        freq_grid[focus_mask],
        (mean_curve - sd_curve)[focus_mask],
        (mean_curve + sd_curve)[focus_mask],
        alpha=0.16,
        color='0.35',
        label='Media ± 1 DE',
    )
    ax.axhline(0.0, lw=0.8, ls='--', color='0.55')
    ax.axvline(5.5, lw=0.8, ls=':', color='0.55')
    ax.axvline(8.0, lw=0.8, ls=':', color='0.55')
    ax.set_xlim(FOCUS_FMIN, FOCUS_FMAX)
    ax.set_title(ch, fontsize=10)
    ax.grid(alpha=0.12)
    ax.spines[['top', 'right']].set_visible(False)

for ax in axes[5:]:
    ax.set_xlabel('Frecuencia (Hz)')
for ax in axes[::5]:
    ax.set_ylabel(r'Peso funcional $\beta_c(f)$')

axes[0].legend(frameon=True, fontsize=8, loc='best')
fig.tight_layout()

fig_weight_focus_path = FIG_DIR / 'funciones_peso_theta_pre_alfa.png'
fig.savefig(fig_weight_focus_path, dpi=300, bbox_inches='tight')
plt.show()
print('Figura focalizada:', fig_weight_focus_path)

# Mapa focalizado: la escala se calcula solo con 4–10 Hz.
beta_mean_focus = beta_mean[:, focus_mask]
vmax_focus = float(np.max(np.abs(beta_mean_focus)))
if vmax_focus == 0:
    vmax_focus = 1.0
norm_focus = TwoSlopeNorm(vmin=-vmax_focus, vcenter=0.0, vmax=vmax_focus)

fig, ax = plt.subplots(figsize=(10.5, 5.2))
im = ax.imshow(
    beta_mean_focus,
    aspect='auto',
    origin='upper',
    extent=[FOCUS_FMIN, FOCUS_FMAX, n_channels - 0.5, -0.5],
    cmap='coolwarm',
    norm=norm_focus,
)
ax.set_yticks(np.arange(n_channels))
ax.set_yticklabels(channels)
ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('Canal')
ax.axvline(5.5, lw=0.9, ls=':', color='black', alpha=0.6)
ax.axvline(8.0, lw=0.9, ls=':', color='black', alpha=0.6)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'Peso funcional medio $\beta_c(f)$')
fig.tight_layout()

fig_map_focus_path = FIG_DIR / 'mapa_funcion_peso_theta_pre_alfa.png'
fig.savefig(fig_map_focus_path, dpi=300, bbox_inches='tight')
plt.show()
print('Mapa focalizado:', fig_map_focus_path)


## 11. Criterios de interpretación y resultados principales

La interpretación de las funciones de peso se apoya en los siguientes criterios:

1. **Consistencia:** A1 + Cholesky + LogReg debe reproducir el desempeño de los cuadernos 03 y 04.
2. **Estabilidad:** buscar regiones donde la forma y el signo de $\beta_c(f)$ sean relativamente consistentes entre pliegues; no interpretar oscilaciones aisladas.
3. **Dirección:** como `PD_OFF=1`, $\beta_c(f)>0$ se asocia condicionalmente con mayor log-odds predicho de PD_OFF y $\beta_c(f)<0$ con HC.
4. **Theta completa primero:** el intervalo 4–8 Hz se mantiene como banda principal. La separación en theta lenta (4–5.5 Hz) y pre-alfa/theta rápida (5.5–8 Hz) es un análisis secundario guiado por la literatura para localizar qué subregión explica un patrón observado en theta.
5. **Pre-alfa no se selecciona por el resultado:** el intervalo 5.5–8 Hz tiene una definición previa en la literatura de Parkinson y ha sido relacionado con enlentecimiento posterior. Por tanto, su análisis debe presentarse como contraste neurofisiológico preespecificado, no como una búsqueda post hoc de la subbanda con mayor peso.
6. **Sensibilidad de borde:** los picos puntuales próximos a 4 o 13 Hz deben interpretarse con cautela si coinciden con amplificación de las bases de borde. La figura completa se conserva, pero las conclusiones principales deben apoyarse en regiones estables, integrales por banda y el control de sensibilidad interior.
7. **Contexto neurofisiológico:** contrastar los patrones estables con la caracterización independiente de PAF/PSD y con la literatura sobre theta, pre-alfa y enlentecimiento posterior. No forzar concordancia si los pesos son inestables.
8. **Alcance:** los pesos de una regresión logística L2 en una muestra de 48 sujetos son asociaciones del modelo, no biomarcadores clínicos validados ni efectos causales.

**Importante:** la magnitud de $\beta_c(f)$ es condicional a todas las demás frecuencias y canales. Debido a la correlación espectral y espacial del EEG, no debe interpretarse como una prueba univariada de que una frecuencia concreta difiere entre grupos.

**Resultados principales.** La transformación inversa reproduce el predictor lineal del clasificador con un error
máximo del orden de 3 × 10⁻¹⁴. El patrón más consistente se observa en el intervalo pre-alfa (5,5–8 Hz): en los siete
canales posteriores (P7, P8, P3, Pz, P4, O1 y O2) la integral firmada de $\beta_c(f)$ sobre ese intervalo es positiva
en las 15 particiones externas. P3 y O1 presentan la mayor concordancia de forma entre particiones. Las amplitudes
próximas a 13 Hz se asocian a la parametrización de borde de la base (número de condición de G ≈ 1,40 × 10³) y no se
utilizan para conclusiones neurofisiológicas.

In [ ]:
expected_files = [
    CONFIG_PATH,
    metric_path,
    prediction_path,
    split_path,
    transform_path,
    beta_path,
    summary_metric_path,
    stability_path,
    basis_diag_path,
    edge_sensitivity_path,
    alpha_interior_path,
    band_fold_path,
    band_summary_path,
    fig_weight_path,
    fig_map_path,
    fig_weight_focus_path,
    fig_map_focus_path,
]

for path in expected_files:
    if not path.exists():
        raise FileNotFoundError(f'No se generó el producto esperado: {path}')

assert len(metric_df) == 15
assert np.isfinite(metric_df[METRICS].to_numpy()).all()
assert transform_df['max_error_logit_functional'].max() < 1e-8
assert reconstruction_error < 1e-10

metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'analysis': 'interpretabilidad_A1_Cholesky_LogReg',
    'n_subjects': int(n_subjects),
    'class_counts': {
        'HC': int((y == 0).sum()),
        'PD_OFF': int((y == 1).sum()),
    },
    'n_external_evaluations': int(len(metric_df)),
    'channels': channels,
    'n_basis': int(n_basis),
    'k_opt': int(k_opt),
    'band_hz': [BAND_FMIN, BAND_FMAX],
    'max_cholesky_reconstruction_error': reconstruction_error,
    'gram_condition_number': gram_condition_number,
    'knot_spacing_hz': knot_spacing,
    'interpretation_interior_hz': [INTERIOR_FMIN, INTERIOR_FMAX],
    'max_functional_logit_error': float(
        transform_df['max_error_logit_functional'].max()
    ),
    'environment': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'scipy': scipy.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'matplotlib': plt.matplotlib.__version__,
    },
}
metadata_path = OUT_DIR / 'metadatos_ejecucion.json'
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8'
)

print('=' * 92)
print('CUADERNO 05 COMPLETADO: PRODUCTOS VERIFICADOS')
print('=' * 92)
print('Configuración interpretada: A1 + Cholesky + Regresión Logística')
print('Clase positiva: PD_OFF')
print('Evaluaciones externas:', len(metric_df))
print('Balanced accuracy:', f"{metric_df['balanced_acc'].mean():.3f} ± {metric_df['balanced_acc'].std(ddof=1):.3f}")
print('AUC:', f"{metric_df['auc'].mean():.3f} ± {metric_df['auc'].std(ddof=1):.3f}")
print('\nProductos:')
for path in expected_files + [metadata_path]:
    print(' -', path)
